In [3]:
import scanpy as sc
import numpy as np
import pyviper
import pandas as pd
import random
import glob 
import os

In [1]:
print("wspr")

wspr


# only thing you change per sample run (this cell below)

In [46]:
data_lungadata = sc.read_h5ad('/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung7/sample1/modified_pipeline_results/adata_lung7_sample1_square008um_stromal.h5ad')

In [4]:
# 1) Load your ARACNe3 network
network_path = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_reimagined/stromal/GRN_stromal/consolidated-net_defaultid.tsv" #change this per tissue
df = pd.read_csv(network_path, sep="\t")
# 2) Convert ARACNe3 table -> pyviper regulon-format (regulator, target, mor, likelihood)
regulon = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df, regul_size=100)
# 3) Wrap as Interactome
network_interactome_lung = pyviper.Interactome("lungtfcotf", regulon) #change this per tissue
print("Number of regulons in interactome:", network_interactome_lung.size())

Number of regulons in interactome: 2112


In [6]:
network_interactome_lung.net_table.head()

,regulator,target,mor,likelihood
2708105,AATF,B2M,-0.290440,1.000000
2706151,AATF,SKIC8,0.207617,0.747344
2707672,AATF,CYP20A1,0.166158,0.701622
2707353,AATF,FAM89B,0.213895,0.696507
2707613,AATF,DENR,0.027758,0.634309


In [11]:
network_path_epithelial = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/stromal/consolidated-net_defaultid.tsv"
df_epithelial = pd.read_csv(network_path_epithelial, sep="\t")
regulon_epithelial = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_epithelial, regul_size=100)
network_interactome_epithelial = pyviper.Interactome("lung_stromal2", regulon_epithelial)

In [ ]:
print("Number of regulons in interactome:", network_interactome_epithelial.size())


Number of regulons in interactome: 6095


In [21]:
import scanpy as sc
import pandas as pd
import pyviper
import time

# ============================================================================
# MANUAL INPUT - EDIT THESE PATHS
# ============================================================================

# Input: Combined h5ad file (WITHOUT barcode prefixes)
input_h5ad = '/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1_tumor/binned_outputs/square_008um/super_updated_again.h5ad'

# Output: Where to save the protein activity h5ad
output_h5ad = '/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad'

# Networks
network_path_stromal = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/stromal/consolidated-net_defaultid.tsv"
network_path_epithelial = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/epithelial/consolidated-net_defaultid.tsv"
network_path_immune = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/immune/consolidated-net_defaultid.tsv"

# ============================================================================
# RUN metaVIPER
# ============================================================================

print("="*80)
print("metaVIPER: Multi-Network Protein Activity Inference")
print("="*80)
start_time = time.time()

# 1) Load data
print(f"\n📂 Loading data from:")
print(f"   {input_h5ad}")
data_lungadata = sc.read_h5ad(input_h5ad)
print(f"   ✅ Loaded: {data_lungadata.n_obs} cells, {data_lungadata.n_vars} genes")
print(f"   📋 First 5 barcodes: {list(data_lungadata.obs_names[:5])}")

# 2) Load networks
print(f"\n📚 Loading networks...")
df_stromal = pd.read_csv(network_path_stromal, sep="\t")
regulon_stromal = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_stromal, regul_size=100)
network_interactome_stromal = pyviper.Interactome("lung_stromal", regulon_stromal)

df_epithelial = pd.read_csv(network_path_epithelial, sep="\t")
regulon_epithelial = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_epithelial, regul_size=100)
network_interactome_epithelial = pyviper.Interactome("lung_epithelial", regulon_epithelial)

df_immune = pd.read_csv(network_path_immune, sep="\t")
regulon_immune = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_immune, regul_size=100)
network_interactome_immune = pyviper.Interactome("lung_immune", regulon_immune)

print(f"   ✅ Loaded 3 networks")

# 3) Filter networks to match genes (CRITICAL - no warnings, faster)
print(f"\n🔧 Filtering networks to match expression data...")
network_interactome_stromal.filter_targets(data_lungadata.var_names)
network_interactome_epithelial.filter_targets(data_lungadata.var_names)
network_interactome_immune.filter_targets(data_lungadata.var_names)
print(f"   ✅ Networks filtered")

# 4) Run metaVIPER
print(f"\n🚀 Running metaVIPER with 32 cores...")
viper_start = time.time()

lung_nes_meta = pyviper.viper(
    gex_data=data_lungadata,
    interactome=[network_interactome_stromal, 
                 network_interactome_epithelial,
                 network_interactome_immune],
    enrichment="area",
    eset_filter=False,
    njobs=32,
    verbose=True
)

viper_time = time.time() - viper_start
print(f"\n✅ metaVIPER completed in {viper_time/60:.2f} minutes")

# 5) Save as h5ad (NO barcode manipulation!)
print(f"\n💾 Saving results...")
print(f"   Output: {output_h5ad}")
print(f"   Format: h5ad")
print(f"   Proteins: {lung_nes_meta.shape[1]}")
print(f"   Cells: {lung_nes_meta.shape[0]}")

# Verify barcodes match input
print(f"\n🔍 Barcode verification:")
print(f"   Input barcodes (first 3): {list(data_lungadata.obs_names[:3])}")
print(f"   Output barcodes (first 3): {list(lung_nes_meta.obs_names[:3])}")

if list(data_lungadata.obs_names) == list(lung_nes_meta.obs_names):
    print(f"   ✅ Barcodes match perfectly!")
else:
    print(f"   ⚠️  WARNING: Barcodes don't match!")

# Save
lung_nes_meta.write_h5ad(output_h5ad)

total_time = time.time() - start_time

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Output saved to: {output_h5ad}")
print(f"Number of proteins: {lung_nes_meta.shape[1]}")
print(f"Number of cells: {lung_nes_meta.shape[0]}")

metaVIPER: Multi-Network Protein Activity Inference

📂 Loading data from:
   /shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1_tumor/binned_outputs/square_008um/super_updated_again.h5ad
   ✅ Loaded: 447595 cells, 18054 genes
   📋 First 5 barcodes: ['s_008um_00301_00321-1', 's_008um_00602_00290-1', 's_008um_00515_00112-1', 's_008um_00789_00234-1', 's_008um_00526_00291-1']

📚 Loading networks...
   ✅ Loaded 3 networks

🔧 Filtering networks to match expression data...
Removed 121095 targets.
Removed 195644 targets.
Removed 121861 targets.
   ✅ Networks filtered

🚀 Running metaVIPER with 32 cores...
Preparing the association scores
Computing regulons enrichment with aREA
0/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
1/3 networks complete.
Rank transforming the data
Preparing the 1-tailed 

/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/pyviper/_viper.py:306: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  op = AnnData(preOp)



✅ metaVIPER completed in 17.76 minutes

💾 Saving results...
   Output: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad
   Format: h5ad
   Proteins: 7060
   Cells: 447595

🔍 Barcode verification:
   Input barcodes (first 3): ['s_008um_00301_00321-1', 's_008um_00602_00290-1', 's_008um_00515_00112-1']
   Output barcodes (first 3): ['s_008um_00000_00175-1', 's_008um_00000_00179-1', 's_008um_00000_00202-1']
   ⚠️  WARNING: Barcodes don't match!

✅ COMPLETE!
Total runtime: 18.88 minutes
Output saved to: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad
Number of proteins: 7060
Number of cells: 447595


In [ ]:
print(/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad)

In [23]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# CONFIG
# ============================================================
FILE_PATH = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad"
OUTPUT_FILE = "lung1_s1_groundtruth.png"
MARKERS = ["AATF"]

def get_spatial_coords(adata):
    """Automatically finds spatial coordinates in the AnnData object."""
    # 1. Check standard obsm keys
    if 'spatial' in adata.obsm:
        print("   ✅ Found coordinates in adata.obsm['spatial']")
        return adata.obsm['spatial']
    
    if 'X_spatial' in adata.obsm:
        print("   ✅ Found coordinates in adata.obsm['X_spatial']")
        return adata.obsm['X_spatial']

    # 2. Check metadata (obs) columns for Visium standard
    if 'array_row' in adata.obs and 'array_col' in adata.obs:
        print("   ✅ Found coordinates in adata.obs['array_row'/'array_col']")
        # Visium: array_col is x, array_row is y (often inverted, but good for scatter)
        return np.column_stack((adata.obs['array_col'].values, adata.obs['array_row'].values))

    # 3. Check generic x/y columns
    if 'x' in adata.obs and 'y' in adata.obs:
        print("   ✅ Found coordinates in adata.obs['x'/'y']")
        return np.column_stack((adata.obs['x'].values, adata.obs['y'].values))
        
    if 'spatial_1' in adata.obs and 'spatial_2' in adata.obs:
        print("   ✅ Found coordinates in adata.obs['spatial_1'/'spatial_2']")
        return np.column_stack((adata.obs['spatial_1'].values, adata.obs['spatial_2'].values))

    # 4. Failure
    print("❌ ERROR: Could not find spatial coordinates.")
    print("   Available obsm keys:", list(adata.obsm.keys()))
    print("   Available obs columns:", list(adata.obs.columns))
    return None

def main():
    print(f"📂 Loading: {FILE_PATH}")
    try:
        adata = sc.read_h5ad(FILE_PATH)
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return

    # 🔍 FIND COORDINATES
    coords = get_spatial_coords(adata)
    if coords is None:
        return

    # Check available markers
    available_markers = [m for m in MARKERS if m in adata.var_names]
    if not available_markers:
        print("❌ None of the requested markers were found in adata.var_names.")
        return

    # Plot
    fig, axes = plt.subplots(1, len(available_markers), figsize=(5 * len(available_markers), 5))
    if len(available_markers) == 1: axes = [axes] 

    print("🎨 Plotting...")
    for i, marker in enumerate(available_markers):
        try:
            expression = adata[:, marker].X
            if hasattr(expression, "toarray"): 
                expression = expression.toarray().flatten()
            else:
                expression = expression.flatten()
        except:
            continue

        ax = axes[i]
        # Invert Y axis for Visium coordinates so it matches the histology image
        scatter = ax.scatter(coords[:, 0], -coords[:, 1], c=expression, s=2, cmap='magma')
        
        ax.set_title(f"{marker} Activity")
        ax.axis('off')
        plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(OUTPUT_FILE, dpi=300)
    print(f"✅ Plot saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

📂 Loading: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad
❌ ERROR: Could not find spatial coordinates.
   Available obsm keys: []
   Available obs columns: ['n_genes']


In [2]:
import scanpy as sc
import pandas as pd
import pyviper

# ============================================================================
# PATHS
# ============================================================================
input_h5ad = '/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1_tumor/modified_pipeline_results/adata_lung1_sample1_square008um_COMBINED.h5ad'

network_path_stromal = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_reimagined/stromal/GRN_stromal/consolidated-net_defaultid.tsv"
network_path_epithelial = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_reimagined/epithelial/GRN_epithelial/consolidated-net_defaultid.tsv"
network_path_immune = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_reimagined/immune/GRN_immune/consolidated-net_defaultid.tsv"

# ============================================================================
# STEP-BY-STEP PROTEIN TRACKING
# ============================================================================

print("="*80)
print("TRACKING WHERE PROTEINS DISAPPEAR")
print("="*80)

# Load data
print("\n📂 Step 1: Loading data...")
data_lungadata = sc.read_h5ad(input_h5ad)
print(f"   Genes in spatial data: {data_lungadata.n_vars}")

# Load networks
print("\n📚 Step 2: Loading networks...")
df_stromal = pd.read_csv(network_path_stromal, sep="\t")
df_epithelial = pd.read_csv(network_path_epithelial, sep="\t")
df_immune = pd.read_csv(network_path_immune, sep="\t")

# Check unique regulators in each network
print("\n🔍 Step 3: Regulators in RAW networks...")
stromal_regs = set(df_stromal['regulator.values'].unique())
epithelial_regs = set(df_epithelial['regulator.values'].unique())
immune_regs = set(df_immune['regulator.values'].unique())

print(f"   Stromal regulators: {len(stromal_regs)}")
print(f"   Epithelial regulators: {len(epithelial_regs)}")
print(f"   Immune regulators: {len(immune_regs)}")

all_regulators = stromal_regs | epithelial_regs | immune_regs
print(f"   🔢 TOTAL unique regulators across all 3 networks: {len(all_regulators)}")

# Check overlap with spatial data
spatial_genes = set(data_lungadata.var_names)
regs_in_spatial = all_regulators & spatial_genes
print(f"   ✅ Regulators present in spatial data: {len(regs_in_spatial)} ({len(regs_in_spatial)/len(all_regulators)*100:.1f}%)")
print(f"   ❌ Regulators MISSING from spatial: {len(all_regulators - spatial_genes)}")

# Create regulons with DIFFERENT regul_size thresholds
print("\n🧪 Step 4: Testing different regul_size thresholds...")

for regul_size in [0, 25, 50, 100]:
    print(f"\n--- regul_size = {regul_size} ---")
    
    # Create regulons
    regulon_stromal = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_stromal, regul_size=regul_size)
    regulon_epithelial = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_epithelial, regul_size=regul_size)
    regulon_immune = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_immune, regul_size=regul_size)
    
    print(f"   After regul_size filter:")
    print(f"      Stromal: {len(regulon_stromal)} regulators")
    print(f"      Epithelial: {len(regulon_epithelial)} regulators")
    print(f"      Immune: {len(regulon_immune)} regulators")
    
    # Create interactomes
    int_stromal = pyviper.Interactome("stromal", regulon_stromal)
    int_epithelial = pyviper.Interactome("epithelial", regulon_epithelial)
    int_immune = pyviper.Interactome("immune", regulon_immune)
    
    before_filter = int_stromal.size() + int_epithelial.size() + int_immune.size()
    
    # Filter to spatial genes
    int_stromal.filter_targets(data_lungadata.var_names)
    int_epithelial.filter_targets(data_lungadata.var_names)
    int_immune.filter_targets(data_lungadata.var_names)
    
    after_filter = int_stromal.size() + int_epithelial.size() + int_immune.size()
    
    print(f"   After filter_targets():")
    print(f"      Stromal: {int_stromal.size()} regulators")
    print(f"      Epithelial: {int_epithelial.size()} regulators")
    print(f"      Immune: {int_immune.size()} regulators")
    print(f"   📊 Total regulators: {before_filter} → {after_filter}")
    
    # Get unique regulators across all 3
    all_regs = set()
    for regulon in [regulon_stromal, regulon_epithelial, regulon_immune]:
        all_regs.update(regulon.keys())
    
    print(f"   🎯 Unique regulators across all 3 networks: {len(all_regs)}")

# Detailed investigation of regul_size=100
print("\n" + "="*80)
print("DETAILED ANALYSIS: regul_size=100")
print("="*80)

regulon_stromal = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_stromal, regul_size=100)
regulon_epithelial = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_epithelial, regul_size=100)
regulon_immune = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_immune, regul_size=100)

print("\n🔬 Checking target counts per regulator (sample)...")
# Show why regulators get filtered
sample_regs = list(regulon_stromal.keys())[:10]
for reg in sample_regs:
    n_targets = len(regulon_stromal[reg]['tfmode'])
    n_in_spatial = sum(1 for t in regulon_stromal[reg]['tfmode'].keys() if t in spatial_genes)
    print(f"   {reg}: {n_targets} total targets → {n_in_spatial} in spatial data")

print("\n💡 CONCLUSION:")
print("   The regul_size parameter REQUIRES each regulator to have")
print("   AT LEAST that many targets in the network.")
print("   After filter_targets(), some regulators may drop below this threshold.")
print("\n   To get MORE proteins, use a LOWER regul_size (e.g., 25 or 50)")
print("   Trade-off: Lower regul_size = more proteins but lower quality")

TRACKING WHERE PROTEINS DISAPPEAR

📂 Step 1: Loading data...
   Genes in spatial data: 18012

📚 Step 2: Loading networks...

🔍 Step 3: Regulators in RAW networks...
   Stromal regulators: 2112
   Epithelial regulators: 2366
   Immune regulators: 2213
   🔢 TOTAL unique regulators across all 3 networks: 2374
   ✅ Regulators present in spatial data: 2284 (96.2%)
   ❌ Regulators MISSING from spatial: 90

🧪 Step 4: Testing different regul_size thresholds...

--- regul_size = 0 ---
   After regul_size filter:
      Stromal: 0 regulators
      Epithelial: 0 regulators
      Immune: 0 regulators
Removed 0 targets.
Removed 0 targets.
Removed 0 targets.
   After filter_targets():
      Stromal: 0 regulators
      Epithelial: 0 regulators
      Immune: 0 regulators
   📊 Total regulators: 0 → 0
   🎯 Unique regulators across all 3 networks: 4

--- regul_size = 25 ---
   After regul_size filter:
      Stromal: 52800 regulators
      Epithelial: 59150 regulators
      Immune: 55325 regulators
Removed

KeyError: 'tfmode'

In [47]:
lung_nes = pyviper.viper(gex_data= data_lungadata, # gene expression signature
                            interactome=network_interactome_lung, # gene regulatory network
                            enrichment = "area",
                            return_as_df=True,
                            njobs=16,
                            verbose=False)

/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/pyviper/_aREA/aREA_classic.py:94: UserWarning: interactome "lungtfcotf" contains 13348 targets missing from gex_data.var.
	Please run interactome.filter_targets(gex_data.var_names) on your network to
	resolve this. It is highly recommend to do this on the unPruned network and
	then prune. This way the Pruned network contains a consistent number of targets per
	regulator, all of which exist within gex_data.
  warnings.warn('interactome "' + str(interactome.name) + '" contains ' +
/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/pyviper/_aREA/aREA_classic.py:94: UserWarning: interactome "lungtfcotf" contains 13348 targets missing from gex_data.var.
	Please run interactome.filter_targets(gex_data.var_names) on your network to
	resolve this. It is highly recommend to do this on the unPruned network and
	then prune. This way the Pruned network contains a consistent number of targets per
	regulator, all 

In [48]:
lung_nes

,AATF,ABCG1,ABLIM3,ABT1,ACAD8,ACTL6B,ACTN2,ACTR5,ADNP,ADNP2,...,ZSWIM3,ZSWIM4,ZSWIM5,ZSWIM6,ZSWIM7,ZUP1,ZXDA,ZXDB,ZXDC,ZZEF1
s_008um_00007_00671-1,-0.016252,-0.205698,0.120197,-0.016670,-0.006160,-0.148882,-0.001994,-0.009632,-0.106813,0.054752,...,0.006329,-0.010133,-0.895457,-0.405939,-0.020610,-0.325828,-0.071084,-0.006278,-0.168572,-0.133464
s_008um_00008_00680-1,-0.013232,-0.871993,-0.010885,-0.013572,0.184375,0.002278,-0.001624,-0.007842,0.046788,0.002309,...,-0.004836,-0.008250,0.004534,-0.041476,-0.016780,0.007674,-0.003901,-0.005111,0.005611,-0.003007
s_008um_00009_00678-1,-0.017647,-0.606697,-0.211668,-0.018100,-0.006688,0.125997,-0.002165,0.129705,-0.002654,0.027167,...,-0.006450,0.085406,0.006047,-0.184006,0.026997,-0.164613,-0.346124,-0.164341,0.165334,0.159484
s_008um_00009_00679-1,-0.018809,-0.469492,-0.015474,0.029271,-0.007129,0.003238,-0.002308,-0.011148,-0.002829,0.003282,...,-0.006875,0.009913,0.006445,0.085322,-0.023853,0.142306,-0.014131,-0.007266,0.007977,0.049734
s_008um_00014_00667-1,-0.010909,-0.740283,-0.189609,-0.039754,-0.004135,0.001878,1.866805,-0.006466,-0.001641,0.001903,...,-0.003988,-0.006802,0.003738,-0.174898,0.092004,-0.238776,-0.013150,-0.148542,0.004626,-0.002479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
s_008um_00710_00235-1,0.044118,-0.951637,-0.176079,-0.017385,-0.006424,0.002918,1.590545,-0.010046,-0.071991,-0.171855,...,-0.006195,-0.010568,0.005808,-0.460410,-0.021495,-0.400691,-0.004998,-0.136094,-0.116068,-0.003852
s_008um_00823_00710-1,-0.011838,-0.225795,-0.258374,-0.012142,-0.004487,0.002038,-0.001453,-0.007016,0.039324,0.002065,...,-0.004327,-0.007381,0.004056,-0.532566,-0.015013,-0.608236,-0.003491,-0.203233,0.005020,-0.002691
s_008um_00826_00686-1,0.040268,-0.547025,-0.173145,-0.014286,-0.005279,0.002398,-0.001709,-0.008255,-0.137934,0.002430,...,-0.005091,-0.008684,0.004773,-0.560972,-0.017664,-0.589379,-0.004107,-0.134568,-0.222466,-0.206907
s_008um_00831_00553-1,-0.014393,-0.934876,-0.011841,-0.044962,-0.005455,0.002478,-0.001766,-0.008531,-0.002165,0.002511,...,-0.005261,-0.008974,0.004932,-0.182894,-0.018253,-0.184657,-0.004244,-0.005560,0.006104,-0.003271


In [49]:
lung_nes.to_csv(
    "/shares/vasciaveo_lab/aarulselvan/arachne/Latest_PA_tables/lung7sample1/stromal/lung_nes.csv"
)

In [20]:
import os
import scanpy as sc
import pandas as pd
import anndata as ad
from pathlib import Path

# =============================================================================
# !!! EDIT THESE 4 LINES FOR EACH DATASET !!!
# =============================================================================
INPUT_PATH  = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung3/sample1/binned_outputs/square_008um/filtered_feature_bc_matrix"
SAMPLE_ID   = "lung3_sample1_square008um"
FILE_TYPE   = "standard_10x"  # "dense_txt", "dense_tsv", "dense_csv", "standard_10x", or "custom_10x"
OUTPUT_H5AD = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung3/sample1/binned_outputs/square_008um/super_updated_again.h5ad"

# =============================================================================
# FLEXIBLE LOAD LOGIC
# =============================================================================
def load_data(path_str, fmt):
    path = Path(path_str)
    print(f"[{SAMPLE_ID}] Loading {fmt} from: {path}...")
    
    try:
        if fmt == "dense_txt":
            adata = sc.read_text(path, delimiter='\t').T
            
        elif fmt == "dense_csv":
            adata = sc.read_csv(path)
            
        elif fmt == "dense_tsv":
            df = pd.read_csv(path, sep='\t', index_col=0)
            adata = ad.AnnData(df.T)
            
        elif fmt == "standard_10x":
            adata = sc.read_10x_mtx(path, var_names='gene_symbols', cache=False)
            
        elif fmt == "custom_10x":
            mtx_files = list(path.glob("*matrix*.mtx*"))
            if not mtx_files: 
                available = [f.name for f in path.glob("*")]
                raise FileNotFoundError(f"No .mtx found. Folder contains: {available}")
            
            print(f"   -> Found matrix: {mtx_files[0].name}")
            adata = sc.read_mtx(mtx_files[0]).T
            
            # Load features/genes
            feat_files = list(path.glob("*features*.tsv*")) or list(path.glob("*genes*.tsv*"))
            if feat_files:
                print(f"   -> Found features: {feat_files[0].name}")
                genes = pd.read_csv(feat_files[0], sep='\t', header=None)
                adata.var_names = genes[1].values if genes.shape[1] > 1 else genes[0].values
            
            # Load barcodes
            bar_files = list(path.glob("*barcodes*.tsv*"))
            if bar_files:
                print(f"   -> Found barcodes: {bar_files[0].name}")
                barcodes = pd.read_csv(bar_files[0], sep='\t', header=None)
                adata.obs_names = barcodes[0].values
                
        else:
            raise ValueError(f"Invalid FILE_TYPE: {fmt}")
        
        # Make names unique
        adata.var_names_make_unique()
        adata.obs_names_make_unique()
        
        print(f"   ✅ Loaded: {adata.n_obs} cells, {adata.n_vars} genes")
        return adata
        
    except Exception as e:
        print(f"❌ ERROR LOADING DATA: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# =============================================================================
# MAIN
# =============================================================================
print("="*80)
print(f"Converting to h5ad: {SAMPLE_ID}")
print("="*80)

# Load data
adata = load_data(INPUT_PATH, FILE_TYPE)

if adata is None:
    print("❌ Failed to load data. Exiting.")
    exit(1)

print(f"\n📋 Initial data:")
print(f"   Shape: {adata.shape}")

# =============================================================================
# PREPROCESSING
# =============================================================================
print(f"\n🔧 Preprocessing...")

# Filter cells and genes
print(f"   Filtering cells (min_genes=200)...")
sc.pp.filter_cells(adata, min_genes=1)
print(f"      -> {adata.n_obs} cells remaining")

print(f"   Filtering genes (min_cells=3)...")
sc.pp.filter_genes(adata, min_cells=1)
print(f"      -> {adata.n_vars} genes remaining")

# Normalization
print(f"   Normalizing to 10,000 counts per cell...")
sc.pp.normalize_total(adata, target_sum=1e4)

print(f"   Log1p transformation...")
sc.pp.log1p(adata)

# Highly variable genes
print(f"   Finding highly variable genes (top 2000)...")
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
hvg_list = adata.var_names[adata.var['highly_variable']].tolist()
print(f"      -> {len(hvg_list)} HVGs identified")

# =============================================================================
# SAVE
# =============================================================================
print(f"\n📋 Final data:")
print(f"   Shape: {adata.shape}")
print(f"   First 5 barcodes: {list(adata.obs_names[:5])}")
print(f"   First 5 genes: {list(adata.var_names[:5])}")
print(f"   Matrix type: {type(adata.X).__name__}")

# Create output directory if needed
os.makedirs(os.path.dirname(OUTPUT_H5AD), exist_ok=True)

# Save as h5ad
print(f"\n💾 Saving to: {OUTPUT_H5AD}")
adata.write_h5ad(OUTPUT_H5AD)

print("\n" + "="*80)
print("✅ CONVERSION COMPLETE!")
print("="*80)
print(f"Sample: {SAMPLE_ID}")
print(f"Output: {OUTPUT_H5AD}")
print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")
print(f"HVGs: {len(hvg_list)}")

Converting to h5ad: lung3_sample1_square008um
[lung3_sample1_square008um] Loading standard_10x from: /shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung3/sample1/binned_outputs/square_008um/filtered_feature_bc_matrix...
   ✅ Loaded: 605471 cells, 18085 genes

📋 Initial data:
   Shape: (605471, 18085)

🔧 Preprocessing...
   Filtering cells (min_genes=200)...
      -> 605431 cells remaining
   Filtering genes (min_cells=3)...
      -> 17796 genes remaining
   Normalizing to 10,000 counts per cell...
   Log1p transformation...
   Finding highly variable genes (top 2000)...
      -> 2000 HVGs identified

📋 Final data:
   Shape: (605431, 17796)
   First 5 barcodes: ['s_008um_00301_00321-1', 's_008um_00602_00290-1', 's_008um_00425_00829-1', 's_008um_00728_00006-1', 's_008um_00526_00291-1']
   First 5 genes: ['SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'PERM1']
   Matrix type: csr_matrix

💾 Saving to: /shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung3/sample1/binned_outputs/sq

In [18]:
import scanpy as sc

# Load both files
print("="*80)
print("COMPARING H5AD FILES")
print("="*80)

file1 = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1_tumor/modified_pipeline_results/adata_lung1_sample1_square008um_COMBINED.h5ad"
file2 = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1_tumor/binned_outputs/square_008um/super_updated_again.h5ad"

print("\n📂 Loading files...")
adata1 = sc.read_h5ad(file1)
print(f"   File 1: COMBINED (from SingleR pipeline)")
adata2 = sc.read_h5ad(file2)
print(f"   File 2: updated_adata (from conversion script)")

print("\n" + "="*80)
print("DIMENSIONS")
print("="*80)
print(f"File 1 (COMBINED):      {adata1.n_obs:>8,} cells × {adata1.n_vars:>6,} genes")
print(f"File 2 (updated_adata): {adata2.n_obs:>8,} cells × {adata2.n_vars:>6,} genes")

print("\n" + "="*80)
print("BARCODES")
print("="*80)
print(f"File 1 first 5: {list(adata1.obs_names[:5])}")
print(f"File 2 first 5: {list(adata2.obs_names[:5])}")

# Check overlap
common_barcodes = set(adata1.obs_names) & set(adata2.obs_names)
print(f"\nCommon barcodes: {len(common_barcodes):,}")
print(f"File 1 only: {len(set(adata1.obs_names) - set(adata2.obs_names)):,}")
print(f"File 2 only: {len(set(adata2.obs_names) - set(adata1.obs_names)):,}")

print("\n" + "="*80)
print("GENES")
print("="*80)
print(f"File 1 first 5: {list(adata1.var_names[:5])}")
print(f"File 2 first 5: {list(adata2.var_names[:5])}")

common_genes = set(adata1.var_names) & set(adata2.var_names)
print(f"\nCommon genes: {len(common_genes):,}")
print(f"File 1 only: {len(set(adata1.var_names) - set(adata2.var_names)):,}")
print(f"File 2 only: {len(set(adata2.var_names) - set(adata1.var_names)):,}")

print("\n" + "="*80)
print("METADATA")
print("="*80)
print(f"File 1 obs columns: {list(adata1.obs.columns)}")
print(f"File 2 obs columns: {list(adata2.obs.columns)}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
if adata1.n_obs == adata2.n_obs and adata1.n_vars == adata2.n_vars:
    print("✅ Dimensions match perfectly!")
elif adata2.n_obs > adata1.n_obs:
    print(f"⚠️  File 2 has MORE cells ({adata2.n_obs - adata1.n_obs:,} extra)")
    print(f"   This is expected - File 1 was filtered by SingleR")
elif adata1.n_obs > adata2.n_obs:
    print(f"⚠️  File 1 has MORE cells ({adata1.n_obs - adata2.n_obs:,} extra)")
    print(f"   Something is wrong - conversion should have more cells")

COMPARING H5AD FILES

📂 Loading files...
   File 1: COMBINED (from SingleR pipeline)
   File 2: updated_adata (from conversion script)

DIMENSIONS
File 1 (COMBINED):       250,220 cells × 18,012 genes
File 2 (updated_adata):  447,595 cells × 18,054 genes

BARCODES
File 1 first 5: ['lung1_sample1_tumor_epi_s_008um_00000_00502-1', 'lung1_sample1_tumor_epi_s_008um_00000_00503-1', 'lung1_sample1_tumor_epi_s_008um_00000_00539-1', 'lung1_sample1_tumor_epi_s_008um_00000_00540-1', 'lung1_sample1_tumor_epi_s_008um_00000_00541-1']
File 2 first 5: ['s_008um_00301_00321-1', 's_008um_00602_00290-1', 's_008um_00515_00112-1', 's_008um_00789_00234-1', 's_008um_00526_00291-1']

Common barcodes: 0
File 1 only: 250,220
File 2 only: 447,595

GENES
File 1 first 5: ['SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'PERM1']
File 2 first 5: ['SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'PERM1']

Common genes: 18,012
File 1 only: 0
File 2 only: 42

METADATA
File 1 obs columns: ['in_tissue', 'n_genes_by_counts', 'log1p_n_gen

In [ ]:
import torch
import torch.nn as nn
import lightning as L
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import h5py
import anndata as ad
import os
import json
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.neighbors import NearestNeighbors
from PIL import Image

# ==========================================
# ⚙️ CONFIGURATION (UPDATED FOR LUNG1 SAMPLE1)
# ==========================================
CONFIG = {
    # 🟢 CHANGED: Pointing to Sample 1 feature file
    "features_path": "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1/features1/uni_features.h5",
    
    # 🟢 CHANGED: Pointing to the NEWPAtables file you specified
    "gt_path": "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad",
    
    # 🟢 CHANGED: Assuming subset structure matches lung1sample1 directory
    "subsets": {
        "Epithelial": "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/epithelial/lung_nes.csv",
        "Immune": "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/immune/lung_nes.csv",
        "Stromal": "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/stromal/lung_nes.csv"
    },

    # 🟢 CHANGED: Switched 'sample2_tumor' to 'sample1' for spatial images
    "image_path": "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1/binned_outputs/square_008um/spatial/tissue_hires_image.png",
    "spatial_dir": "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1/binned_outputs/square_008um/spatial",
    
    # Keeping model checkpoint same (assuming you want to test the same model)
    "model_ckpt": "/shares/vasciaveo_lab/aarulselvan/arachne/DeepSpot_adhiban/lightning_logs/version_29/checkpoints/rmse-best.ckpt",
    "protein_list_path": "/shares/vasciaveo_lab/aarulselvan/arachne/pyviper_regulators.txt",
    "target_protein": "NKX2-1",
    "device": "cuda",
    "batch_size": 16384
}

# ... [MODEL CLASSES] ...
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU())
    def forward(self, x): return x + self.block(x)

class RobustMLP(L.LightningModule):
    def __init__(self, num_targets=100, input_dim=5120, hidden_dim=1024, dropout=0.3):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout), ResidualBlock(hidden_dim, dropout), ResidualBlock(hidden_dim, dropout), nn.Linear(hidden_dim, num_targets))
    def forward(self, x): return self.net(x)

def run_fast_correct_norm():
    print(f"🚀 Starting FAST & CORRECT Debug (All Cell Types) for {CONFIG['target_protein']}...")

    # ==========================================
    # 1. LOAD ALL PRECOMPUTED FEATURES
    # ==========================================
    print("\n📦 Loading .h5 Features...")
    try:
        with h5py.File(CONFIG['features_path'], 'r') as f:
            feat_key = next(k for k in f.keys() if k in ['features', 'matrix', 'embeddings', 'uni'])
            bc_key = next(k for k in f.keys() if k in ['barcodes', 'index', 'obs_names'])
            features = np.array(f[feat_key])
            barcodes = np.array(f[bc_key]).astype(str)
    except Exception as e:
        print(f"❌ Error loading features from {CONFIG['features_path']}: {e}")
        return

    print(f"   ⚖️  Global Normalization on {len(features):,} spots...")
    scaler = StandardScaler()
    features_norm = scaler.fit_transform(features)
    
    clean_bcs = pd.Index(barcodes).str.split('-').str[0]
    feat_df = pd.DataFrame(features_norm, index=clean_bcs)
    feat_df = feat_df[~feat_df.index.duplicated()]

    # ==========================================
    # 2. LOAD METADATA (POSITIONS & GT)
    # ==========================================
    print("\n📍 Loading Coordinates & Ground Truth...")
    
    # Load Positions
    pos_file = os.path.join(CONFIG['spatial_dir'], "tissue_positions.parquet")
    if not os.path.exists(pos_file): pos_file = os.path.join(CONFIG['spatial_dir'], "tissue_positions.csv")
    
    try:
        if pos_file.endswith('.parquet'): pos_df = pd.read_parquet(pos_file)
        else: pos_df = pd.read_csv(pos_file, header=None, names=["barcode", "in", "r", "c", "pxl_row", "pxl_col"])
        
        if "barcode" in pos_df.columns: pos_df = pos_df.set_index("barcode")
        
        pos_df.index = pos_df.index.astype(str)
        pos_df['clean_bc'] = pos_df.index.str.split('-').str[0]
        pos_df = pos_df.drop_duplicates('clean_bc').set_index('clean_bc')
    except Exception as e:
        print(f"❌ Error loading positions: {e}")
        return

    # Load GT (All Cells)
    try:
        adata = ad.read_h5ad(CONFIG['gt_path'])
        adata.obs_names_make_unique()
        adata.obs_names = adata.obs_names.astype(str)
        adata.obs['clean_bc'] = adata.obs_names.str.split('-').str[0]
    except Exception as e:
        print(f"❌ Error loading GT from {CONFIG['gt_path']}: {e}")
        return

    # ==========================================
    # 3. INTERSECT (MASTER SET)
    # ==========================================
    print("\n🔗 Aligning 'Universe' of Valid Data...")
    # Intersection of: Features + Positions + GT (regardless of cell type)
    common_all = sorted(list(set(feat_df.index)
                          .intersection(pos_df.index)
                          .intersection(adata.obs['clean_bc'])))
    
    print(f"   ✅ Total Valid Spots (All Types): {len(common_all):,}")
    if len(common_all) == 0: raise ValueError("No valid spots found!")

    # Extract Data for ALL common spots
    target_pos = pos_df.loc[common_all]
    target_feats = feat_df.loc[common_all].values 
    
    # Get GT Values
    adata = adata[adata.obs['clean_bc'].isin(common_all)]
    gt_map = dict(zip(adata.obs['clean_bc'], adata[:, CONFIG['target_protein']].X.flatten()))
    gt_values_all = np.array([gt_map[bc] for bc in common_all])

    # Get Coords
    try:
        with open(os.path.join(CONFIG['spatial_dir'], "scalefactors_json.json")) as f:
            sf = json.load(f)["tissue_hires_scalef"]
    except:
        print("⚠️ Scale factors not found, using default 1.0")
        sf = 1.0
    
    raw_x = target_pos["pxl_col_in_fullres"].values
    raw_y = target_pos["pxl_row_in_fullres"].values
    coords = np.stack([raw_x, raw_y], axis=1)
    
    plot_x = raw_x * sf
    plot_y = raw_y * sf

    # ==========================================
    # 4. PREDICT (ONCE FOR ALL)
    # ==========================================
    print("\n🔮 Predicting on full slide...")
    # Build Graph on ALL valid spots
    nbrs = NearestNeighbors(n_neighbors=5, algorithm='ball_tree').fit(coords)
    _, indices = nbrs.kneighbors(coords)
    
    feat_tensor = torch.from_numpy(target_feats).float()
    neighbor_feats = [feat_tensor[indices[:, k]] for k in range(5)]
    final_input = torch.cat(neighbor_feats, dim=1)

    # Run Model
    model = RobustMLP.load_from_checkpoint(CONFIG['model_ckpt'], strict=False).eval().to(CONFIG['device'])
    with open(CONFIG['protein_list_path']) as f: all_p = [l.strip() for l in f if l.strip()]
    
    if CONFIG['target_protein'] in all_p:
        p_idx = all_p.index(CONFIG['target_protein'])
    else:
        print(f"❌ Protein {CONFIG['target_protein']} not found in regulator list.")
        return

    preds = []
    final_input = final_input.to(CONFIG['device'])
    with torch.no_grad():
        for i in range(0, len(final_input), CONFIG['batch_size']):
            out = model(final_input[i : i+CONFIG['batch_size']])
            preds.append(out[:, p_idx].cpu())
    
    pred_values_all = torch.cat(preds).numpy()

    # ==========================================
    # 5. SUBSET EVALUATION & PLOTTING
    # ==========================================
    print("\n📊 Evaluating Cell Types...")
    
    # Setup Plot: 3 Rows (Epi, Immune, Stromal), 2 Cols (GT, Pred)
    fig, axes = plt.subplots(3, 2, figsize=(18, 24), dpi=150)
    try:
        img_bg = Image.open(CONFIG['image_path']).convert("RGB")
    except:
        print("⚠️ Image not found, plotting without background.")
        img_bg = None
    
    colors = ["#000080", "#FFFFFF", "#FFD700"]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom", colors)
    
    # Calculate global limits for consistent coloring across all plots
    global_lim = np.percentile(np.abs(np.concatenate([gt_values_all, pred_values_all])), 98)

    # Dictionary to map 'common_all' indices to barcodes for easy lookup
    bc_to_idx = {bc: i for i, bc in enumerate(common_all)}

    for i, (ctype, csv_path) in enumerate(CONFIG['subsets'].items()):
        print(f"   👉 Processing {ctype}...")
        
        try:
            # Load Subset Barcodes
            sub_df = pd.read_csv(csv_path, index_col=0)
            sub_bcs = set(sub_df.index.astype(str).str.split('-').str[0])
            
            # Find indices in 'common_all' that belong to this subset
            valid_bcs = [bc for bc in common_all if bc in sub_bcs]
            if not valid_bcs:
                print(f"      ⚠️ No matching spots for {ctype}. Skipping.")
                continue
                
            subset_indices = [bc_to_idx[bc] for bc in valid_bcs]
            
            # Slice Data
            y_true = gt_values_all[subset_indices]
            y_pred = pred_values_all[subset_indices]
            x_plot = plot_x[subset_indices]
            y_plot = plot_y[subset_indices]
            
            # Metrics
            mse = mean_squared_error(y_true, y_pred)
            corr, _ = pearsonr(y_true, y_pred)
            print(f"      MSE: {mse:.6f} | R: {corr:.4f}")

            # Plot GT
            ax_gt = axes[i, 0]
            if img_bg: ax_gt.imshow(img_bg)
            sc1 = ax_gt.scatter(x_plot, y_plot, c=y_true, cmap=cmap, s=3, vmin=-global_lim, vmax=global_lim)
            ax_gt.set_title(f"{ctype} - GT", fontsize=16, fontweight='bold')
            ax_gt.axis('off')
            plt.colorbar(sc1, ax=ax_gt, shrink=0.7)

            # Plot Pred
            ax_pred = axes[i, 1]
            if img_bg: ax_pred.imshow(img_bg)
            sc2 = ax_pred.scatter(x_plot, y_plot, c=y_pred, cmap=cmap, s=3, vmin=-global_lim, vmax=global_lim)
            ax_pred.set_title(f"{ctype} - Pred\nMSE={mse:.4f}, R={corr:.2f}", fontsize=16, fontweight='bold')
            ax_pred.axis('off')
            plt.colorbar(sc2, ax=ax_pred, shrink=0.7)
            
        except Exception as e:
            print(f"      ⚠️ Error processing {ctype}: {e}")

    plt.tight_layout()
    save_name = f"lung1sample1_eval_{CONFIG['target_protein']}.png"
    plt.savefig(save_name, bbox_inches='tight')
    print(f"\n✅ Saved comparison plot to {save_name}")

if __name__ == "__main__":
    run_fast_correct_norm()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import json
import numpy as np
import matplotlib.image as mpimg

# --- CONFIG ---
visium_root = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung3/sample1/binned_outputs/square_008um"
h5ad_path = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad"

# Note: We use the PNG now, not the TIFF
img_path = f"{visium_root}/spatial/tissue_hires_image.png"
parquet_path = f"{visium_root}/spatial/tissue_positions.parquet"
sf_path = f"{visium_root}/spatial/scalefactors_json.json"

# 1. LOAD DATA
print("Loading Metadata...")
adata = sc.read_h5ad(h5ad_path)
pos_df = pd.read_parquet(parquet_path)
if "barcode" in pos_df.columns: 
    pos_df = pos_df.set_index("barcode")

# Filter: Only plot spots that are in your dataset (unique barcodes only)
unique_barcodes = adata.obs_names.unique()
common_barcodes = pos_df.index.intersection(unique_barcodes)
pos_df = pos_df.loc[common_barcodes]

print(f"Plotting {len(pos_df)} spots on Hires Image.")

# 2. CALCULATE COORDINATES
with open(sf_path, "r") as f: 
    sf = json.load(f)

# Standard SpaceRanger Logic: FullRes_Coord * Hires_ScaleFactor
# We usually DO NOT need manual shifts for the 'hires' image because 
# SpaceRanger crops this image specifically to match these coordinates.
hires_scale = sf['tissue_hires_scalef']

x_coords = pos_df["pxl_col_in_fullres"] * hires_scale
y_coords = pos_df["pxl_row_in_fullres"] * hires_scale

# 3. PLOT
print("Loading PNG Image...")
img = mpimg.imread(img_path)

print("Generating Overlay...")
plt.figure(figsize=(10, 10), dpi=150)

# Show Image
plt.imshow(img)

# Plot Spots
plt.scatter(x_coords, 
            y_coords, 
            s=1,         # Small spot size for HD
            c='red', 
            alpha=0.5,   # Transparency to see tissue underneath
            linewidth=0)

plt.title(f"Coverage Check: {len(pos_df)} Bins on Hires Image")
plt.axis('off')
plt.show()

Loading Metadata...


KeyboardInterrupt: 

In [1]:
import scanpy as sc
import pandas as pd

# Path to your lung3 sample1 h5ad
file_path = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad"

# Load the file
adata = sc.read_h5ad(file_path)

# Extract the 10x10 slice
# Note: This handles both sparse and dense matrices
slice_10x10 = adata[:10, :10].X
if hasattr(slice_10x10, "toarray"):
    slice_10x10 = slice_10x10.toarray()

# Convert to DataFrame for a clean print with labels
df_slice = pd.DataFrame(
    slice_10x10, 
    index=adata.obs_names[:10], 
    columns=adata.var_names[:10]
)

print(f"--- 10x10 Slice of {file_path} ---")
print(df_slice)

--- 10x10 Slice of /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad ---
                           AAMP      AATF     ABCA1    ABCA12     ABCA3  \
s_008um_00000_00175-1 -0.066510 -0.001962 -0.003155 -0.000259 -0.001344   
s_008um_00000_00179-1  0.045880 -0.000749 -0.004144 -0.000340 -0.001766   
s_008um_00000_00202-1 -0.054242 -0.022162  0.067744  0.072297 -0.002952   
s_008um_00000_00251-1  0.056287 -0.000808 -0.000622 -0.000107 -0.000554   
s_008um_00000_00283-1  0.031844 -0.001308  0.000466 -0.000173 -0.000896   
s_008um_00000_00333-1 -0.003873 -0.000500 -0.000804 -0.000066 -0.000343   
s_008um_00000_00392-1  0.015358  0.017960 -0.004701 -0.000386 -0.002003   
s_008um_00000_00410-1  0.044469 -0.055272  0.065260 -0.000381  0.020462   
s_008um_00000_00413-1 -0.004589 -0.003347  0.031661 -0.000442 -0.002293   
s_008um_00000_00472-1 -0.002163  0.010452  0.065953 -0.000208 -0.001081   

                          ABCA7     ABCA8     ABCB1     ABCB4     AB

In [7]:
import scanpy as sc
import numpy as np
from scipy.stats import norm
import pandas as pd

# --- CONFIG ---
input_file = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad"
output_file = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update2_mLog10.h5ad"

# 1. Load Data
print(f"📂 Loading: {input_file}")
vp_data = sc.read_h5ad(input_file)

# Ensure data is dense for the math operations
if hasattr(vp_data.X, "toarray"):
    vp_data.X = vp_data.X.toarray()

# 2. Explicit Transform
print("🧪 Applying mLog10 transform...")
# Note: This highlights significantly activated regulators (Right Tail)
vp_data.layers['mLog10'] = -1 * np.log10(norm.sf(vp_data.X))

# 3. Analyze Transform
# We use .values if it's a layer to get the underlying numpy array
mlog_values = vp_data.layers['mLog10']

# Handle potential inf/nan for counting
valid_values = mlog_values[np.isfinite(mlog_values)]

count_gt_2 = np.sum(valid_values > 1)
count_lt_2 = np.sum(valid_values < 1)

print("\n📊 Statistical Summary:")
print(f"   - Values > 2 (Highly Significant): {count_gt_2:,}")
print(f"   - Values < 2:                      {count_lt_2:,}")

# 4. Save
print(f"\n💾 Saving to: {output_file}")
vp_data.write_h5ad(output_file)

# 5. Verify with 10x10 print
df_10x10 = pd.DataFrame(
    vp_data.layers['mLog10'][:10, :10], 
    index=vp_data.obs_names[:10], 
    columns=vp_data.var_names[:10]
)
print("\n--- 10x10 Sample Output ---")
print(df_10x10)

📂 Loading: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad
🧪 Applying mLog10 transform...

📊 Statistical Summary:
   - Values > 2 (Highly Significant): 35,093
   - Values < 2:                      3,159,985,607

💾 Saving to: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update2_mLog10.h5ad

--- 10x10 Sample Output ---
                           AAMP      AATF     ABCA1    ABCA12     ABCA3  \
s_008um_00000_00175-1  0.278590  0.300351  0.299938  0.300940  0.300564   
s_008um_00000_00179-1  0.317221  0.300770  0.299596  0.300912  0.300418   
s_008um_00000_00202-1  0.282639  0.293418  0.325144  0.326811  0.300008   
s_008um_00000_00251-1  0.320975  0.300750  0.300815  0.300993  0.300838   
s_008um_00000_00283-1  0.312205  0.300577  0.301191  0.300970  0.300720   
s_008um_00000_00333-1  0.299690  0.300857  0.300751  0.301007  0.300911   
s_008um_00000_00392-1  0.306384  0.307298  0.299404  0.300896  0.300336   
s_008um_00000_00410-1  0

In [ ]:
vp_data.layers['mLog10'] = -1 * np.log10(norm.sf(vp_data.X))

In [ ]:
import scanpy as sc
import pandas as pd
import pyviper
import time

# ============================================================================
# MANUAL INPUT - EDIT THESE PATHS
# ============================================================================

# Input: Combined h5ad file (WITHOUT barcode prefixes)
input_h5ad = '/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung6/sample1/binned_outputs/square_008um/super_updated_again.h5ad'

# Output: Where to save the protein activity h5ad
output_h5ad = '/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung6sample1/update_NES.h5ad'

# Networks
network_path_stromal = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/stromal/consolidated-net_defaultid.tsv"
network_path_epithelial = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/epithelial/consolidated-net_defaultid.tsv"
network_path_immune = "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/immune/consolidated-net_defaultid.tsv"

# ============================================================================
# RUN metaVIPER
# ============================================================================

print("="*80)
print("metaVIPER: Multi-Network Protein Activity Inference")
print("="*80)
start_time = time.time()

# 1) Load data
print(f"\n📂 Loading data from:")
print(f"   {input_h5ad}")
data_lungadata = sc.read_h5ad(input_h5ad)
print(f"   ✅ Loaded: {data_lungadata.n_obs} cells, {data_lungadata.n_vars} genes")
print(f"   📋 First 5 barcodes: {list(data_lungadata.obs_names[:5])}")

# 2) Load networks
print(f"\n📚 Loading networks...")
df_stromal = pd.read_csv(network_path_stromal, sep="\t")
regulon_stromal = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_stromal, regul_size=100)
network_interactome_stromal = pyviper.Interactome("lung_stromal", regulon_stromal)

df_epithelial = pd.read_csv(network_path_epithelial, sep="\t")
regulon_epithelial = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_epithelial, regul_size=100)
network_interactome_epithelial = pyviper.Interactome("lung_epithelial", regulon_epithelial)

df_immune = pd.read_csv(network_path_immune, sep="\t")
regulon_immune = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df_immune, regul_size=100)
network_interactome_immune = pyviper.Interactome("lung_immune", regulon_immune)

print(f"   ✅ Loaded 3 networks")

# 3) Filter networks to match genes (CRITICAL - no warnings, faster)
print(f"\n🔧 Filtering networks to match expression data...")
network_interactome_stromal.filter_targets(data_lungadata.var_names)
network_interactome_epithelial.filter_targets(data_lungadata.var_names)
network_interactome_immune.filter_targets(data_lungadata.var_names)
print(f"   ✅ Networks filtered")

# 4) Run metaVIPER
print(f"\n🚀 Running metaVIPER with 32 cores...")
viper_start = time.time()

lung_nes_meta = pyviper.viper(
    gex_data=data_lungadata,
    interactome=[network_interactome_stromal, 
                 network_interactome_epithelial,
                 network_interactome_immune],
    enrichment="area",
    eset_filter=False,
    njobs=32,
    verbose=True
)

viper_time = time.time() - viper_start
print(f"\n✅ metaVIPER completed in {viper_time/60:.2f} minutes")

# 5) Save as h5ad (NO barcode manipulation!)
print(f"\n💾 Saving results...")
print(f"   Output: {output_h5ad}")
print(f"   Format: h5ad")
print(f"   Proteins: {lung_nes_meta.shape[1]}")
print(f"   Cells: {lung_nes_meta.shape[0]}")

# Verify barcodes match input
print(f"\n🔍 Barcode verification:")
print(f"   Input barcodes (first 3): {list(data_lungadata.obs_names[:3])}")
print(f"   Output barcodes (first 3): {list(lung_nes_meta.obs_names[:3])}")

if list(data_lungadata.obs_names) == list(lung_nes_meta.obs_names):
    print(f"   ✅ Barcodes match perfectly!")
else:
    print(f"   ⚠️  WARNING: Barcodes don't match!")

# Save
lung_nes_meta.write_h5ad(output_h5ad)

total_time = time.time() - start_time

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Output saved to: {output_h5ad}")
print(f"Number of proteins: {lung_nes_meta.shape[1]}")
print(f"Number of cells: {lung_nes_meta.shape[0]}")

metaVIPER: Multi-Network Protein Activity Inference

📂 Loading data from:
   /shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung6/sample1/binned_outputs/square_008um/super_updated_again.h5ad
   ✅ Loaded: 311885 cells, 17916 genes
   📋 First 5 barcodes: ['s_008um_00602_00290-1', 's_008um_00526_00291-1', 's_008um_00681_00396-1', 's_008um_00128_00278-1', 's_008um_00052_00559-1']

📚 Loading networks...
   ✅ Loaded 3 networks

🔧 Filtering networks to match expression data...
Removed 123291 targets.
Removed 198814 targets.
Removed 123875 targets.
   ✅ Networks filtered

🚀 Running metaVIPER with 32 cores...
Preparing the association scores
Computing regulons enrichment with aREA


/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/pyviper/_viper.py:306: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  op = AnnData(preOp)


0/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
1/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
2/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
3/3 networks complete.
Integrating NES matrices together with mvws=1...

✅ metaVIPER completed in 24.33 minutes

💾 Saving results...
   Output: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung6sample1/update_NES.h5ad
   Format: h5ad
   Proteins: 7060
   Cells: 311885

🔍 Barcode verification:
   I

0/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
1/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
2/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
3/3 networks complete.
Integrating NES matrices together with mvws=1...
0/3 networks complete.
Rank transforming the data
Preparing the 1-tailed / 2-tailed matrices
Computing the likelihood matrix
Computing the modes matrix
Computing 2-tail enrichment
Computing 1-tail enrichment
Integrating enrichment
1/3 

In [5]:
import pandas as pd
import pyviper

# ---------------- CONFIG ----------------
network_paths = {
    "stromal":    "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/stromal/consolidated-net_defaultid.tsv",
    "epithelial": "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/epithelial/consolidated-net_defaultid.tsv",
    "immune":     "/shares/vasciaveo_lab/data/adhiban_scRNAseq_datasets/lung/lung_again/immune/consolidated-net_defaultid.tsv"
}

interactomes = {}

# ---------------- CREATE INTERACTOMES ----------------
for name, path in network_paths.items():
    print(f"📚 Loading {name} network...")
    df = pd.read_csv(path, sep="\t")
    
    regulon = pyviper.pp.aracne3_to_regulon(net_file=None, net_df=df, regul_size=100)
    interactome = pyviper.Interactome(f"lung_{name}", regulon)
    
    interactomes[name] = interactome
    print(f"   ✅ {name} interactome created with {len(interactome.net_table)} edges")

print("\n🎉 All interactomes created!")


📚 Loading stromal network...
   ✅ stromal interactome created with 609500 edges
📚 Loading epithelial network...
   ✅ epithelial interactome created with 699300 edges
📚 Loading immune network...
   ✅ immune interactome created with 651800 edges

🎉 All interactomes created!


In [6]:
# Make sure your epithelial interactome object is loaded
# network_interactome_epithelial

protein = "AATF"

# Filter the net_table for any edges where AATF is source or target
aatf_interactions = network_interactome_epithelial.net_table[
    (network_interactome_epithelial.net_table['source'] == protein) |
    (network_interactome_epithelial.net_table['target'] == protein)
]

# Print the number of interactions
print(f"AATF has {len(aatf_interactions)} interactions in the epithelial network.")

# Optional: view the first few interactions
print(aatf_interactions.head())


NameError: name 'network_interactome_epithelial' is not defined

In [3]:
# Count interactions where AATF is either source or target
aatf_interactions = network_interactome_epithelial.net_table[
    (network_interactome_epithelial.net_table['source'] == 'AATF') |
    (network_interactome_epithelial.net_table['target'] == 'AATF')
]

print(f"Number of interactions involving AATF: {len(aatf_interactions)}")

# Optional: see the first few interactions
print(aatf_interactions.head())


NameError: name 'network_interactome_epithelial' is not defined

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import json
import numpy as np
import matplotlib.image as mpimg

# --- CONFIG ---
visium_root = "/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung1/sample1/binned_outputs/square_008um"
h5ad_path = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample1/update_NES.h5ad"

# Note: We use the PNG now, not the TIFF
img_path = f"{visium_root}/spatial/tissue_hires_image.png"
parquet_path = f"{visium_root}/spatial/tissue_positions.parquet"
sf_path = f"{visium_root}/spatial/scalefactors_json.json"

# 1. LOAD DATA
print("Loading Metadata...")
adata = sc.read_h5ad(h5ad_path)
pos_df = pd.read_parquet(parquet_path)
if "barcode" in pos_df.columns: 
    pos_df = pos_df.set_index("barcode")

# Filter: Only plot spots that are in your dataset (unique barcodes only)
unique_barcodes = adata.obs_names.unique()
common_barcodes = pos_df.index.intersection(unique_barcodes)
pos_df = pos_df.loc[common_barcodes]

print(f"Plotting {len(pos_df)} spots on Hires Image.")

# 2. CALCULATE COORDINATES
with open(sf_path, "r") as f: 
    sf = json.load(f)

# Standard SpaceRanger Logic: FullRes_Coord * Hires_ScaleFactor
# We usually DO NOT need manual shifts for the 'hires' image because 
# SpaceRanger crops this image specifically to match these coordinates.
hires_scale = sf['tissue_hires_scalef']

x_coords = pos_df["pxl_col_in_fullres"] * hires_scale
y_coords = pos_df["pxl_row_in_fullres"] * hires_scale

# 3. PLOT
print("Loading PNG Image...")
img = mpimg.imread(img_path)

print("Generating Overlay...")
plt.figure(figsize=(10, 10), dpi=150)

# Show Image
plt.imshow(img)

# Plot Spots
plt.scatter(x_coords, 
            y_coords, 
            s=1,         # Small spot size for HD
            c='red', 
            alpha=0.5,   # Transparency to see tissue underneath
            linewidth=0)

plt.title(f"Coverage Check: {len(pos_df)} Bins on Hires Image")
plt.axis('off')
plt.show()

In [8]:
print("hello")

hello


In [ ]:
import scanpy as sc
import numpy as np
from scipy.stats import norm
import pandas as pd

# --- CONFIG ---
input_file = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample2/update_NES.h5ad"
output_file = "/shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample2/update_mLog10_2sided.h5ad"

print(f"📂 Loading: {input_file}")
vp_data = sc.read_h5ad(input_file)

# ensure dense numpy array
if hasattr(vp_data.X, "toarray"):
    X = vp_data.X.toarray()
else:
    X = np.array(vp_data.X)

# -------------------------------------------------------
# Two-sided p-value and mLog10
# p_two_sided = 2 * P(Z >= |X|)
# mLog10_2sided = -log10(p_two_sided)
# signed_mLog10 = sign(X) * mLog10_2sided
# -------------------------------------------------------
print("🧪 Computing two-sided p-values and -log10 transforms...")

# numeric safety: avoid exact zeros in p (clip)
p_two_sided = 2.0 * norm.sf(np.abs(X))
p_two_sided = np.clip(p_two_sided, 1e-300, 1.0)   # floor to avoid -log10(inf)

mlog2s = -np.log10(p_two_sided)                  # positive numbers: larger => more significant
signed_mlog = np.sign(X) * mlog2s                # sign encodes activation/repression

# attach layers
vp_data.layers["mLog10_2sided"] = mlog2s
vp_data.layers["signed_mLog10"] = signed_mlog

# analyze
valid_vals = mlog2s[np.isfinite(mlog2s)]
count_gt2 = np.sum(valid_vals > 2)
count_lt2 = np.sum(valid_vals <= 2)

signed_vals = signed_mlog[np.isfinite(signed_mlog)]
count_act = np.sum(signed_vals > 2)
count_rep = np.sum(signed_vals < -2)

print("\n📊 Statistical Summary (two-sided):")
print(f"   - mLog10_2sided > 2 (p < 1e-2): {count_gt2:,}")
print(f"   - mLog10_2sided <= 2:            {count_lt2:,}")

print("\n📊 Signed summary (|signed_mLog10| > 2):")
print(f"   - Activated (signed_mLog10 > 2):  {count_act:,}")
print(f"   - Repressed (signed_mLog10 < -2): {count_rep:,}")

# Save
print(f"\n💾 Saving to: {output_file}")
vp_data.write_h5ad(output_file)

# 10x10 sample (signed_mLog10)
df_10x10 = pd.DataFrame(
    vp_data.layers["signed_mLog10"][:10, :10],
    index=vp_data.obs_names[:10],
    columns=vp_data.var_names[:10]
)
print("\n--- 10x10 Sample Output (signed_mLog10) ---")
print(df_10x10)


📂 Loading: /shares/vasciaveo_lab/aarulselvan/arachne/NEWPAtables/lung1sample2/update_NES.h5ad
🧪 Computing two-sided p-values and -log10 transforms...


In [3]:
#run only for scRNAseq, and only per tissue switch (GRN)
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import pyviper
from scipy import sparse as sp

def generate_metacells_final_v3(adata_path, out_dir, n_target=10):
    print(f"Loading data: {adata_path}")
    adata = sc.read_h5ad(adata_path)
    adata.var_names_make_unique()
    
    # ---------------------------------------------------------
    # NEW: Subsample to 20,000 cells for speed and ARACNe accuracy
    # ---------------------------------------------------------
    target_cells = 50000
    if adata.n_obs > target_cells:
        print(f"Dataset has {adata.n_obs} cells. Subsampling to {target_cells}...")
        sc.pp.subsample(adata, n_obs=target_cells, random_state=42)
    else:
        print(f"Dataset has {adata.n_obs} cells (no subsampling needed).")
    
    # Pre-processing for PCA space
    print("Normalizing and running PCA...")
    adata_raw = adata.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver='randomized')
    pyviper.pp.corr_distance(adata)

    all_metacells = []
    
    # TREAT ENTIRE ADATA AS ONE BATCH
    sub_adata = adata
    sub_raw = adata_raw
    n_cells = sub_adata.n_obs

    print(f"Processing single batch dataset with {n_cells} cells...")

    # If batch is large enough, try the optimized algorithm
    if n_cells > 60:
        try:
            pyviper.pp.repr_metacells(
                sub_adata,
                counts=sub_raw,
                pca_slot='X_pca',
                dist_slot='corr_dist',
                n_cells_per_metacell=n_target,
                perc_data_to_use=None,
                size=500,
                min_median_depth=None,
                perc_incl_data_reused=None,
                key_added="temp_mc",
                njobs=1,
                verbose=False
            )
            all_metacells.append(sub_adata.uns["temp_mc"])
        except Exception as e:
            print(f"Pyviper failed, using fallback: {e}")
            n_mc = max(1, int(n_cells / n_target))
            indices = np.arange(n_cells)
            groups = np.array_split(indices, n_mc)
            for i, g_idx in enumerate(groups):
                mc_val = np.sum(sub_raw.X[g_idx, :], axis=0)
                if sp.issparse(mc_val): mc_val = mc_val.toarray()
                df = pd.DataFrame(mc_val.reshape(1, -1), index=[f"mc_{i}"], columns=adata.var_names)
                all_metacells.append(df)
    else:
        n_mc = max(1, int(n_cells / n_target))
        indices = np.arange(n_cells)
        groups = np.array_split(indices, n_mc)
        for i, g_idx in enumerate(groups):
            mc_val = np.sum(sub_raw.X[g_idx, :], axis=0)
            if sp.issparse(mc_val): mc_val = mc_val.toarray()
            df = pd.DataFrame(mc_val.reshape(1, -1), index=[f"mc_{i}"], columns=adata.var_names)
            all_metacells.append(df)

    # Combine
    final_mc_df = pd.concat(all_metacells)
    
    # Final cleanup and normalization for ARACNe
    mc_adata = ad.AnnData(X=final_mc_df.values, 
                          obs=pd.DataFrame(index=final_mc_df.index), 
                          var=pd.DataFrame(index=final_mc_df.columns))
    sc.pp.normalize_total(mc_adata, target_sum=1e4)
    sc.pp.log1p(mc_adata)

    # ARACNe format: Transpose so Genes are Rows
    expr_df = pd.DataFrame(mc_adata.X, index=mc_adata.obs_names, columns=mc_adata.var_names).T

    out_path = os.path.join(out_dir, "lung6sample2v2.tsv")
    expr_df.to_csv(out_path, sep="\t")
    
    print(f"\n✅ SUCCESS!")
    print(f"Matrix: {expr_df.shape[0]} genes x {expr_df.shape[1]} metacells")
    print(f"Saved to: {out_path}")

if __name__ == "__main__":
    generate_metacells_final_v3(
        adata_path="/shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung6/sample2/binned_outputs/square_008um/super_updated_again.h5ad",
        out_dir="/shares/vasciaveo_lab/aarulselvan/arachne/novelGRNs/IPF"
    )

Loading data: /shares/vasciaveo_lab/aarulselvan/arachne/datasets/lung/lung6/sample2/binned_outputs/square_008um/super_updated_again.h5ad
Dataset has 133872 cells. Subsampling to 50000...
Normalizing and running PCA...


/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:216: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersions']
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:13<00:00,  3.70it/s]


Processing single batch dataset with 50000 cells...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 109.72it/s]
/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/aarulselvan/miniconda3/envs/myenv/lib/python3.10/site-packages/numpy/core/_methods.py:223: RuntimeWarning: invalid value encountered in


✅ SUCCESS!
Matrix: 17945 genes x 500 metacells
Saved to: /shares/vasciaveo_lab/aarulselvan/arachne/novelGRNs/IPF/lung6sample2v2.tsv
